In [1]:
1+1

2

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [3]:
## Determenistic approch for guardrails

def determenistic_guard_rail(text:str)->bool:
    """Returns True if content is blocked"""
    banned_words = ["hack","malware","exploit"]
    return any(kw in text.lower() for kw in banned_words)

test_inputs = [
    "How do I hack into a database?",
    "What is the capital of France?",
    "Explain how malware spreads",
]
print("===Determenistic guardrail approach===")
for inp in test_inputs:
    blocked = determenistic_guard_rail(inp)
    status = "Blocked" if blocked else "Allowed"
    print(f"{status}:{inp}")

===Determenistic guardrail approach===
Blocked:How do I hack into a database?
Allowed:What is the capital of France?
Blocked:Explain how malware spreads


In [4]:
from langchain_groq import ChatGroq

def model_based_guardrail(text:str)->str:
    """ uses a llm for the safety,Returns SAFE OR UNSAFE"""
    model = ChatGroq(model = "llama-3.3-70b-versatile",temperature = 0)
    prompt = f"""is the following input is safe to process? Reply with
    SAFE OR UNSAFE
    Input:{text}"""
    
    result = model.invoke([{"role":"user","content":prompt}])
    return result.content.strip()

print("===Model Based Guardrail===")
for inp in test_inputs:
    verdict = model_based_guardrail(inp)
    status = "UNSAFE"if "UNSAFE" in verdict else "SAFE"
    print(f"{status}:{inp}")

    

===Model Based Guardrail===
UNSAFE:How do I hack into a database?
SAFE:What is the capital of France?
SAFE:Explain how malware spreads


In [20]:
## Langchain has inbuilt middleware PII personally identifiable information

from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain.tools import tool

@tool
def customer_records(query:str)-> str:
    """Look up customer information"""
    return f"Customer record found for query: {query}"

agent = create_agent(
    model = "groq:llama-3.3-70b-versatile",
    tools = [customer_records],
    middleware=[
        PIIMiddleware(
            "email",
            strategy='redact',
            apply_to_input=True
        ),
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True
        ),
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True
        ),
    ],
)
print("Agent with PII middleware created sucessfully")

Agent with PII middleware created sucessfully


In [21]:
response = agent.invoke({"messages":[{"role":"user","content":"My email is john.doe@example.com and my card is 5105-1051-0510-5100. Can you help me?"}]})
response["messages"][-1].content

"I've located your customer record. Is there something specific you'd like to know or change regarding your account?"

In [22]:
response

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and my card is ****-****-****-5100. Can you help me?', additional_kwargs={}, response_metadata={}, id='24d3d0e9-e994-4cb1-be50-b062b18b2302'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '3979ezzsk', 'function': {'arguments': '{"query":"[REDACTED_EMAIL] and card ****-****-****-5100"}', 'name': 'customer_records'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 236, 'total_tokens': 266, 'completion_time': 0.069475159, 'completion_tokens_details': None, 'prompt_time': 0.01134514, 'prompt_tokens_details': None, 'queue_time': 0.161270746, 'total_time': 0.080820299}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fda7d-f45b-7b42-ba97-4d45304930b3-0', tool_calls=[{'name': 'customer_records', 'args': {'que

In [25]:
try:
    response = agent.invoke({"messages":[{"role":"user","content":"here is my  api_key is sk-aphgxwvsyhuehhbhsgyzgyudg223"}]})
except Exception as e:
    print(f"Blocked as ecxcepted{e}")
    

In [19]:
response

{'messages': [HumanMessage(content='my api key is sk-aphgxwvsyhuehhbhsgyzgyudg223', additional_kwargs={}, response_metadata={}, id='13bc9d48-15d7-4bb7-a649-8ef26bedfd6e'),
  AIMessage(content="I can't store or use your API key. Is there something else I can help you with?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 228, 'total_tokens': 249, 'completion_time': 0.066548176, 'completion_tokens_details': None, 'prompt_time': 0.016502515, 'prompt_tokens_details': None, 'queue_time': 0.052078143, 'total_time': 0.083050691}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fda7d-74aa-7b53-a10f-92824ee1759f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 228, 'output_tokens': 21, 'total_tokens': 249})]}

In [34]:
## Human in the loop middleware 
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.tools import tool
@tool
def search_web(query:str)-> str:
    """search the web for information"""
    return f"search results for :{query}"

@tool
def send_mail(to:str,subject:str,body:str)->str:
    """send an email to the receiptent"""
    return f"the emial succesfullt send to :{to}"

@tool
def delete_records(table:str,condition:str)->str:
    """delete records from the database"""
    return f"Deleted records from the table {table} where {condition}"

hit_agent = create_agent(
    model = "groq:llama-3.3-70b-versatile",
    tools=[search_web,send_mail,delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_mail":True,
                "delete_records":True,
                "search_web":False
            }
        )
    ],
    checkpointer=InMemorySaver()
)

print("Agent succesfully created with human in the loop middleware")




Agent succesfully created with human in the loop middleware


In [35]:
config = {"configurable":{"thread_id":"1"}}

In [36]:
result = hit_agent.invoke({"messages":[{"role":"user","content":"send an email to the john@email.com about project details"}]},config=config)
print("Agent paused awaiting for human approval")
print(result)

Agent paused awaiting for human approval
{'messages': [HumanMessage(content='send an email to the john@email.com about project details', additional_kwargs={}, response_metadata={}, id='40bfab13-bb37-483a-8042-a412e9e95449'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '2bb9f25pg', 'function': {'arguments': '{"body":"Please find the project details in the attached file or below.","subject":"Project Details","to":"john@email.com"}', 'name': 'send_mail'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 347, 'total_tokens': 387, 'completion_time': 0.141942533, 'completion_tokens_details': None, 'prompt_time': 0.044357261, 'prompt_tokens_details': None, 'queue_time': 0.051190158, 'total_time': 0.186299794}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fda97-ff7a-7090-9

In [37]:
approved_result = hit_agent.invoke(
    Command(resume={"decisions":[{"type":"approve"}]}),
    config=config
)

print("Approved Final Response")
approved_result["messages"][-1].content

Approved Final Response


''